# SCP Stage A — Self-Collapse Probe PoC

Goal: **under which settings, and by how much, does translation quality collapse when the model is trained on its own outputs?**

Follows Algorithm 1 (Self-Collapse Probe stage):

1. Generate self-translations `T` on a subset of EN sources.
2. Measure `q_before = QE(src, T)` (reference-free COMET-Kiwi).
3. Attach a fresh aggressive **Probe LoRA** (small rank, high lr, no regularization).
4. Train on `(src, T)` pairs — response-only loss, same format as stage3_it SFT.
5. Regenerate `T'`; measure `q_after = QE(src, T')`.
6. Compute `dQE(s) = q_before - q_after`. Large positive `dQE` = vulnerable sentence.
7. Sweep `(rank, learning_rate, num_train_epochs)` and compare collapse distributions.

`data/test.csv` is used as the source corpus because it has human-verified KO references, which lets us cross-check the QE signal against reference-based metrics. For production, `alwaysgood/sec-10k-pre-processed` and `alwaysgood/earnings_call_mono` will be swapped in via `configs/data/*.yaml` — no code changes.

Prompts reuse `configs/prompts/translation_dynamic_5.yaml` (copied from stage3_it) so the model sees the same input distribution it was SFT-trained on.

## 1. Bootstrap environment

In [ ]:
import os, subprocess, sys

def pip_install(args, optional=False):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + list(args)
    print('running:', ' '.join(cmd))
    try:
        subprocess.check_call(cmd)
        return True
    except Exception as e:
        if optional:
            print(f'[WARN] optional install failed: {args} -> {type(e).__name__}: {e}')
            return False
        raise

pip_install(['unsloth', 'unsloth-zoo', 'datasets', 'peft', 'hydra-core', 'omegaconf', 'pandas', 'tqdm', 'matplotlib', 'pyyaml>=6.0.2'])
pip_install(['transformers==5.5.4', 'trl>=0.15.0', '--no-deps'])
pip_install(['sacrebleu'], optional=True)

# IMPORTANT — COMET compatibility note.
# unbabel-comet 2.2.7 (latest) pins `transformers>=4.17,<5.0`, so it is
# INCOMPATIBLE with transformers==5.5.4 required by unsloth. Installing it
# in this env would downgrade transformers and break unsloth.
#
# Instead, create a SEPARATE venv for COMET and export its python path:
#   python3 -m venv ~/.venvs/comet
#   ~/.venvs/comet/bin/pip install 'unbabel-comet>=2.2.7'
#   export COMET_PYTHON=$HOME/.venvs/comet/bin/python
# The QE backend (src/qe.py) auto-detects $COMET_PYTHON and calls COMET via
# subprocess. If COMET_PYTHON is unset, qe.py falls back to a no-op scorer
# (zeros) and the notebook still runs — you just lose the QE signal.
print('bootstrap install done. Restart kernel if this was a fresh environment.')
print('COMET_PYTHON =', os.environ.get('COMET_PYTHON', '(unset — see note above)'))

In [ ]:
import os, sys, gc, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import transformers

warnings.filterwarnings('ignore')
print('torch', torch.__version__, 'transformers', transformers.__version__, 'cuda', torch.cuda.is_available())

# Locate repo root (notebooks/ -> parent)
NB_DIR = Path.cwd().resolve()
REPO_ROOT = NB_DIR if (NB_DIR / 'configs' / 'poc_scp_probe.yaml').exists() else NB_DIR.parent
assert (REPO_ROOT / 'configs' / 'poc_scp_probe.yaml').exists(), f'Could not find repo root from {NB_DIR}'
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print('REPO_ROOT', REPO_ROOT)

## 2. Compose Hydra config

Change `DATA` below to `sec_10k_mono` or `earnings_call_mono` once full-scale training is ready. `MODEL` picks which stage3 checkpoint to probe. All other hparams live in `configs/`.

In [ ]:
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf

# 기본: qwen35 + stage3_it test.csv
MODEL = os.environ.get('MODEL_CONFIG', 'qwen35_4b_it')
MODEL_REPO = os.environ.get('MODEL_REPO', 'alwaysgood/qwen35-st2')
DATA  = os.environ.get('DATA_CONFIG',  'testcsv')
DATA_CSV_PATH = os.environ.get('DATA_CSV_PATH', 'data/test.csv')
PROBE_SUBSET_SIZE = int(os.environ.get('PROBE_SUBSET_SIZE', '200'))  # small for PoC

with initialize_config_dir(version_base=None, config_dir=str(REPO_ROOT / 'configs')):
    cfg = compose(
        config_name='poc_scp_probe',
        overrides=[
            f'model={MODEL}',
            f'data={DATA}',
            f'probe.probe_subset_size={PROBE_SUBSET_SIZE}',
            f'model.pretrained_model_name_or_path={MODEL_REPO}',
            f'data.path={DATA_CSV_PATH}',
        ],
    )
OmegaConf.resolve(cfg)
print('MODEL_CONFIG =', MODEL)
print('MODEL_REPO   =', MODEL_REPO)
print('DATA_CONFIG  =', DATA)
print('DATA_CSV_PATH=', DATA_CSV_PATH)
print(OmegaConf.to_yaml(cfg, resolve=True))


## 3. Build run context (load model, data, QE scorer)

In [ ]:
# unsloth must be imported before transformers patching
import unsloth  # noqa: F401

from src.scp_a import (
    build_context,
    generate_self_translations,
    score_qe_primary,
    score_qe_reference,
    attach_and_train_probe,
    reload_base_model,
)
from src.common import free_vram

ctx = build_context(cfg, load_qe=True)
print(f'records total={len(ctx.records)} probe_subset={len(ctx.probe_records)}')
print(f'qe_primary={ctx.qe_primary.name} qe_reference={ctx.qe_reference.name if ctx.qe_reference else None}')

## 4. Baseline: pre-probe self-translations + QE

`q_before(s)` for every sentence in the probe subset. These are the numbers we will try to collapse.

In [ ]:
t_before = generate_self_translations(ctx, ctx.probe_records)
q_before = score_qe_primary(ctx, ctx.probe_records, t_before)
print('qe_before: mean=%.4f std=%.4f n=%d' % (float(np.mean(q_before)), float(np.std(q_before)), len(q_before)))

before_df = pd.DataFrame([
    {
        'sample_id': r['sample_id'],
        'source_text': r['source_text'],
        'reference_text': r.get('reference_text', ''),
        'sector': r.get('metadata', {}).get('sector', ''),
        'source_type': r.get('metadata', {}).get('source_type', ''),
        'hypothesis_before': t['hypothesis'],
        'qe_before': q,
    }
    for r, t, q in zip(ctx.probe_records, t_before, q_before)
])
before_df.head()

## 5. Single probe run (default config)

Attach a fresh probe LoRA with `configs/probe/aggressive.yaml` defaults, train, regenerate, re-score.

In [ ]:
probe_metrics = attach_and_train_probe(ctx, t_before)
print('probe metrics:', {k: v for k, v in probe_metrics.items() if k != 'losses'})

t_after = generate_self_translations(ctx, ctx.probe_records)
q_after = score_qe_primary(ctx, ctx.probe_records, t_after)
print('qe_after : mean=%.4f std=%.4f n=%d' % (float(np.mean(q_after)), float(np.std(q_after)), len(q_after)))

In [ ]:
run_df = before_df.copy()
run_df['hypothesis_after'] = [t['hypothesis'] for t in t_after]
run_df['qe_after']  = q_after
run_df['delta_qe']  = run_df['qe_before'] - run_df['qe_after']

delta = run_df['delta_qe'].to_numpy()
print('delta_qe: mean=%.4f median=%.4f p90=%.4f max=%.4f' % (
    float(np.mean(delta)), float(np.median(delta)),
    float(np.quantile(delta, 0.9)), float(np.max(delta)),
))
for thr in (0.01, 0.02, 0.05, 0.1):
    share = float(np.mean(delta > thr))
    print(f'  share with delta_qe > {thr:.2f}: {share:.3f} ({int(share * len(delta))} / {len(delta)})')
run_df.nlargest(10, 'delta_qe')[['sample_id', 'sector', 'source_type', 'qe_before', 'qe_after', 'delta_qe']]

## 6. Hyperparameter sweep

For each `(rank, learning_rate, num_train_epochs)` combo from `configs/probe/sweep.yaml`:

- reload base model (independence across combos),
- regenerate `t_before` once per model reload (shared across combos of the same base since the base is the same — we cache),
- train probe, regenerate `t_after`, score.

We reuse the baseline `q_before` captured above (it only depends on the base model and the prompt, which don't change across combos).

In [ ]:
sweep_cfg = OmegaConf.load(REPO_ROOT / 'configs' / 'probe' / 'sweep.yaml')
grid = OmegaConf.to_container(sweep_cfg.grid, resolve=True)

combos = [
    {'rank': r, 'learning_rate': float(lr), 'num_train_epochs': int(e)}
    for r in grid['rank']
    for lr in grid['learning_rate']
    for e in grid['num_train_epochs']
]
print(f'total combos: {len(combos)}')
for c in combos[:3]: print(' ', c)
print('  ...')

In [ ]:
# Cache the baseline so we only generate q_before once per base model.
baseline_hyp = [t['hypothesis'] for t in t_before]
baseline_qe  = list(q_before)

sweep_rows = []
sweep_sentence_rows = []

for combo_idx, combo in enumerate(combos):
    print('=' * 80)
    print(f'[{combo_idx + 1}/{len(combos)}] combo: {combo}')
    print('=' * 80)
    # Reset to the base model for each combo.
    if combo_idx > 0:
        reload_base_model(ctx)

    # Train probe on the cached baseline translations.
    t_before_cached = [{'hypothesis': h} for h in baseline_hyp]
    metrics = attach_and_train_probe(ctx, t_before_cached, probe_overrides=combo)

    # Regenerate with probe adapter attached.
    t_after_c = generate_self_translations(ctx, ctx.probe_records)
    q_after_c = score_qe_primary(ctx, ctx.probe_records, t_after_c)

    delta_c = np.asarray(baseline_qe) - np.asarray(q_after_c)
    sweep_rows.append({
        **combo,
        'n': len(delta_c),
        'qe_before_mean': float(np.mean(baseline_qe)),
        'qe_after_mean': float(np.mean(q_after_c)),
        'delta_mean': float(np.mean(delta_c)),
        'delta_median': float(np.median(delta_c)),
        'delta_p90': float(np.quantile(delta_c, 0.9)),
        'delta_max': float(np.max(delta_c)),
        'share_gt_0.02': float(np.mean(delta_c > 0.02)),
        'share_gt_0.05': float(np.mean(delta_c > 0.05)),
        'probe_final_loss': metrics.get('final_loss'),
        'probe_steps': metrics.get('global_steps'),
        'probe_n_samples': metrics.get('n_samples'),
    })
    for rec, before_q, after_q, after_t in zip(ctx.probe_records, baseline_qe, q_after_c, t_after_c):
        sweep_sentence_rows.append({
            **combo,
            'sample_id': rec['sample_id'],
            'sector': rec.get('metadata', {}).get('sector', ''),
            'source_type': rec.get('metadata', {}).get('source_type', ''),
            'qe_before': float(before_q),
            'qe_after': float(after_q),
            'delta_qe': float(before_q - after_q),
            'hypothesis_after': after_t['hypothesis'],
        })

sweep_df = pd.DataFrame(sweep_rows)
sweep_sentence_df = pd.DataFrame(sweep_sentence_rows)
out_dir = REPO_ROOT / 'artifacts' / 'poc_scp_probe' / 'sweep'
out_dir.mkdir(parents=True, exist_ok=True)
sweep_df.to_csv(out_dir / 'sweep_summary.csv', index=False)
sweep_sentence_df.to_csv(out_dir / 'sweep_sentences.csv', index=False)
print('saved sweep artifacts to', out_dir)
sweep_df

## 7. Analysis

### 7.1 Which settings drop performance the most?

In [ ]:
import matplotlib.pyplot as plt

pivot = sweep_df.pivot_table(
    index=['rank', 'num_train_epochs'],
    columns='learning_rate',
    values='delta_mean',
    aggfunc='mean',
)
print('mean delta_qe by (rank, epochs) x lr:')
display(pivot.round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for (r, e), g in sweep_df.groupby(['rank', 'num_train_epochs']):
    axes[0].plot(g['learning_rate'], g['delta_mean'], marker='o', label=f'r={r}, E={e}')
    axes[1].plot(g['learning_rate'], g['share_gt_0.05'], marker='o', label=f'r={r}, E={e}')
axes[0].set_xscale('log'); axes[0].set_xlabel('probe lr'); axes[0].set_ylabel('mean delta_qe')
axes[0].set_title('Collapse magnitude'); axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3)
axes[1].set_xscale('log'); axes[1].set_xlabel('probe lr'); axes[1].set_ylabel('share(delta_qe > 0.05)')
axes[1].set_title('Vulnerable-sentence rate'); axes[1].legend(fontsize=7); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 7.2 Per-sentence collapse distribution (best combo)

In [ ]:
best = sweep_df.sort_values('share_gt_0.05', ascending=False).head(3)
print('top combos by share(delta_qe > 0.05):')
display(best)

best_combo = best.iloc[0][['rank', 'learning_rate', 'num_train_epochs']].to_dict()
mask = (
    (sweep_sentence_df['rank'] == best_combo['rank'])
    & (sweep_sentence_df['learning_rate'] == best_combo['learning_rate'])
    & (sweep_sentence_df['num_train_epochs'] == best_combo['num_train_epochs'])
)
best_sent = sweep_sentence_df.loc[mask]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(best_sent['delta_qe'], bins=40)
axes[0].axvline(0.0, color='k', linestyle='--', alpha=0.5)
axes[0].set_xlabel('delta_qe'); axes[0].set_ylabel('count')
axes[0].set_title(f'delta_qe distribution (best combo {best_combo})')
axes[0].grid(alpha=0.3)

by_sector = best_sent.groupby('sector')['delta_qe'].agg(['mean', 'count']).sort_values('mean', ascending=False)
axes[1].barh(by_sector.index.astype(str), by_sector['mean'])
axes[1].set_xlabel('mean delta_qe'); axes[1].set_title('collapse by sector (best combo)')
axes[1].grid(alpha=0.3, axis='x')
plt.tight_layout(); plt.show()

print('most-collapsed sentences:')
display(best_sent.nlargest(10, 'delta_qe')[['sample_id', 'sector', 'source_type', 'qe_before', 'qe_after', 'delta_qe']])

### 7.3 Cross-check with reference-based metric

Because test.csv has gold Korean references, we can verify that the **reference-free** collapse signal correlates with a **reference-based** metric drop. If they correlate, dQE is a valid proxy for actual quality loss.

In [ ]:
if ctx.qe_reference is not None and any(r.get('reference_text') for r in ctx.probe_records):
    # Score against gold references — before and after for the best combo.
    best_mask = mask  # reuse from 7.2
    best_after_hyps = best_sent.set_index('sample_id')['hypothesis_after'].to_dict()
    before_by_id = {r['sample_id']: t['hypothesis'] for r, t in zip(ctx.probe_records, t_before)}

    triplets_before, triplets_after, ids = [], [], []
    for r in ctx.probe_records:
        ref = r.get('reference_text', '')
        if not ref:
            continue
        ids.append(r['sample_id'])
        triplets_before.append((r['source_text'], before_by_id[r['sample_id']], ref))
        triplets_after.append((r['source_text'], best_after_hyps.get(r['sample_id'], ''), ref))

    cref_before = ctx.qe_reference.score(triplets_before)
    cref_after  = ctx.qe_reference.score(triplets_after)
    cref_df = pd.DataFrame({
        'sample_id': ids,
        'comet_ref_before': cref_before,
        'comet_ref_after':  cref_after,
        'delta_comet_ref':  np.asarray(cref_before) - np.asarray(cref_after),
    })
    merged = best_sent.merge(cref_df, on='sample_id', how='inner')

    corr_p = merged[['delta_qe', 'delta_comet_ref']].corr(method='pearson').iloc[0, 1]
    corr_s = merged[['delta_qe', 'delta_comet_ref']].corr(method='spearman').iloc[0, 1]
    print(f'pearson(delta_qe, delta_comet_ref) = {corr_p:.3f}')
    print(f'spearman(delta_qe, delta_comet_ref) = {corr_s:.3f}')

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(merged['delta_qe'], merged['delta_comet_ref'], alpha=0.5)
    ax.axhline(0, color='k', alpha=0.3); ax.axvline(0, color='k', alpha=0.3)
    ax.set_xlabel('delta_qe (reference-free)'); ax.set_ylabel('delta_comet_ref (reference-based)')
    ax.set_title('QE collapse signal vs. gold-reference quality drop'); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print('reference scorer unavailable or no references in the subset; skipping cross-check.')

## 8. Takeaways (fill in after running)

- **Which combo collapses hardest?** → check `best_combo` from section 7.2.
- **How much drop is realistic?** → `delta_mean` and `share_gt_0.05` columns in the sweep summary.
- **Does the signal agree with gold references?** → correlation in section 7.3.
- **δ threshold calibration** → pick the `delta_qe` cutoff that retains the top-`rho`% of sentences; this is the `delta` used in Stage B of the full algorithm.

Once this is characterized, Stage B (teacher correction of vulnerable sentences) and Stage C (main LoRA update) are implemented on top of `src/scp_a.py` — the probe harness is reusable.